# VLLM Document Extraction Test (`Qwen/Qwen3-VL-8B-Instruct`)

This notebook tests the structured extraction pipeline using the Vision LLM endpoint (`http://localhost:8002/v1/chat/completions`).

## 1. Setup Environment, Force Reload Modules & Instantiate Extractor

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import json
import importlib

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

# Force reload modules so Jupyter kernel never uses stale cached bytecode
import engine.extractors.pdf_extractor
import engine.extractors.ocr_extractor
import engine.extractors.image_extractor
import engine.extractors.vision_extractor
import engine.extractors.router
import engine.providers.vllm_provider

importlib.reload(engine.extractors.pdf_extractor)
importlib.reload(engine.extractors.ocr_extractor)
importlib.reload(engine.extractors.image_extractor)
importlib.reload(engine.extractors.vision_extractor)
importlib.reload(engine.extractors.router)
importlib.reload(engine.providers.vllm_provider)

from engine.extractors.vision_extractor import VisionExtractor
from engine.extractors.router import ExtractorRouter

t0 = time.time()
vision_extractor = VisionExtractor(role="vision_model")
print(f"All extractor modules freshly reloaded in {time.time() - t0:.3f}s:")
print(f"Provider: {vision_extractor.provider.__class__.__name__}")
print(f"Model: {vision_extractor.provider.model}")
print(f"Endpoint: {vision_extractor.provider.chat_url}")

## 2. Locate Target Document (`Kebijakan MK3L & Komitmen management.pdf` or Sample Image)

In [ ]:
target_rel = os.path.join("test_submissions", "marine", "1 Kepemimpinan dan Komitmen", "Management Visit", "Permohonan Izin Management Visit - Kalijapat", "WhatsApp Image 2026-01-22 at 09.57.19 (1).jpeg")
sample_file = os.path.join(ROOT_DIR, target_rel)

if not os.path.exists(sample_file):
    marine_dir = os.path.join(ROOT_DIR, "test_submissions", "marine")
    for root, _, files in os.walk(marine_dir):
        for f in files:
            if f.lower().endswith((".pdf", ".jpeg", ".jpg", ".png")):
                sample_file = os.path.join(root, f)
                break
        if os.path.exists(sample_file):
            break

print(f"Target file selected: {sample_file}")
print(f"File exists: {os.path.exists(sample_file)}")

## 3. Execute ExtractorRouter & Print Full Text Preview

In [ ]:
router = ExtractorRouter()
t1 = time.time()
doc_data = router.extract_document(sample_file)
duration = time.time() - t1

print(f"[Extraction Completed in {duration:.3f}s]")
print("Status:", doc_data.get("status"))
print("Document Type:", doc_data.get("document_type"))
print("Extractor Used:", doc_data.get("extractor_used"))
print("Vision Model Used:", doc_data.get("vision_model_used"))
print("Characters Extracted:", doc_data.get("characters_extracted"))
print("Vision Confidence:", doc_data.get("vision_confidence"))

print("\n--- EXTRACTED TEXT PREVIEW (First 600 chars) ---")
full_text = "\n\n".join([str(c) for c in doc_data.get("evidence", [])])
print(full_text[:600] + ("..." if len(full_text) > 600 else ""))

## 4. Direct Vision Structured Extraction Test (`Qwen/Qwen3-VL-8B-Instruct`)

In [ ]:
t2 = time.time()
structured_res = vision_extractor.extract(sample_file, ocr_output=doc_data.get("evidence"), metadata=doc_data.get("metadata"))
vis_dur = time.time() - t2

print(f"[Direct Vision Extraction Completed in {vis_dur:.3f}s]")
print(json.dumps(structured_res, indent=2))